# Portfolio API — Zerodha (`slug: zerodha`)

Exercises all `/portfolio/*` endpoints scoped to the `zerodha` source.

**Auth pre-req:** Log in to [kite.zerodha.com](https://kite.zerodha.com) inside the AlphaForge Anton Chrome session (`--remote-debugging-port=9299`). Set `ZERODHA_USER_ID` in `backend/.env.cred.local`.

**What one sync fetches:**
- Equity & ETF via `/oms/portfolio/holdings` — classified by instrument type from the Kite instruments master.
- COIN mutual funds via `/api/mf/holdings` — returned as `asset_class: mutual_fund`. Silently skipped if no COIN holdings.

In [ ]:
import json, os
from pathlib import Path

SLUG     = "zerodha"
MODE     = "http"          # "in_process" | "http"
BASE     = "http://localhost:8000/api/v1"
FIXTURES = Path.cwd().parent / "tests" / "fixtures" / "broker_csvs"

# Dev defaults from backend/app/core/config.py — override via env if you've
# changed AlphaForge Anton admin credentials.
AF_USERNAME = os.getenv("AF_USERNAME", "admin")
AF_PASSWORD = os.getenv("AF_PASSWORD", "alphaforge-anton-dev")

if MODE == "in_process":
    from fastapi.testclient import TestClient
    from app.main import app
    client = TestClient(app)
    PREFIX = "/api/v1"
else:
    import httpx
    client = httpx.Client(base_url=BASE, timeout=60.0)
    PREFIX = ""


def _login() -> str:
    r = client.post(
        f"{PREFIX}/auth/token",
        data={"username": AF_USERNAME, "password": AF_PASSWORD},
    )
    if r.status_code != 200:
        raise RuntimeError(
            f"Auth failed ({r.status_code}): {r.text}. "
            "Set AF_USERNAME / AF_PASSWORD env vars if you changed admin creds."
        )
    return r.json()["access_token"]


def _ensure_auth() -> None:
    if "Authorization" not in client.headers:
        client.headers["Authorization"] = f"Bearer {_login()}"


def _request(method: str, path: str, **kw):
    _ensure_auth()
    r = client.request(method, f"{PREFIX}{path}", **kw)
    if r.status_code == 401:
        client.headers["Authorization"] = f"Bearer {_login()}"
        r = client.request(method, f"{PREFIX}{path}", **kw)
    return r.status_code, r.json() if r.headers.get("content-type", "").startswith("application/json") else r.text


def get(path, **kw):  return _request("GET", path, **kw)
def post(path, **kw): return _request("POST", path, **kw)

def pp(obj):
    print(json.dumps(obj, indent=2, default=str))

_ensure_auth()
print(f"Mode: {MODE}  slug: {SLUG}  authed as: {AF_USERNAME}")

## 1. Source info

`status: ready` when `ZERODHA_USER_ID` is set, `unconfigured` otherwise.

In [ ]:
status, body = get(f"/portfolio/sources/{SLUG}")
print(status)
pp(body)

## 2. Sync

Triggers CDP login → enctoken → Kite OMS holdings fetch. Enctoken cached in `.cache/brokers/zerodha.json`.

> Requires `MODE="http"` with a live server and an open Chrome session.

In [ ]:
status, body = post(f"/portfolio/sources/{SLUG}/sync")
print(status, f"  holdings={body.get('holdings_count')}  status={body.get('info', {}).get('status')}")
for h in (body.get("holdings") or [])[:8]:
    print(f"  {h['asset_class']:12} {h['symbol']:20} qty={h['quantity']:<8}  ltp=₹{h['last_price']:>10,.2f}  pnl={h['pnl_pct']:>+.1f}%")

## 3. Upload CSV (offline fallback)

Skips Chrome — upload a CSV from console.zerodha.com directly.

In [ ]:
csv_path = FIXTURES / "zerodha_holdings.csv"
if csv_path.exists():
    with csv_path.open("rb") as f:
        r = client.post(
            f"{PREFIX}/portfolio/sources/{SLUG}/upload",
            files={"file": (csv_path.name, f, "text/csv")},
        )
    print(r.status_code)
    body = r.json()
    print(f"Uploaded {body.get('holdings_count')} holdings")
else:
    print(f"No fixture at {csv_path} — drop a Zerodha CSV export there.")

## 4. Holdings — zerodha only

In [ ]:
status, body = get("/portfolio/holdings", params={"source": SLUG})
print(status, "  totals:", body.get("totals"))
print(f"\n{len(body.get('holdings', []))} holdings:")
for h in body.get("holdings", []):
    print(f"  [{h['asset_class']:12}] {h['symbol']:20} avg=₹{h['avg_price']:>10,.2f}  ltp=₹{h['last_price']:>10,.2f}  pnl={h['pnl_pct']:>+.1f}%")

## 5. Allocation (zerodha)

In [ ]:
status, body = get("/portfolio/holdings", params={"source": SLUG})
print("Allocation:")
for a in body.get("allocation", []):
    print(f"  {a['asset_class']:12} ₹{a['value']:>14,.0f}  ({a['pct']:>5.1f}%)")

## 6. Treemap (zerodha)

In [ ]:
status, body = get("/portfolio/treemap", params={"source": SLUG})
print(status)
for c in (body.get("cells") or [])[:10]:
    print(f"  {c['symbol']:14} {c['pct']:>5.1f}% @ ({c['left_pct']:>5.1f}, {c['top_pct']:>5.1f}) {c['width_pct']:>5.1f}x{c['height_pct']:>5.1f}")

## 7. Rebalance (zerodha)

In [ ]:
status, body = get("/portfolio/rebalance", params={"source": SLUG})
print("Drift:")
for d in body.get("drift", []):
    print(f"  {d['asset_class']:12} target {d['target_pct']:>5.1f}% · actual {d['actual_pct']:>5.1f}% · drift {d['drift_pct']:>+5.1f}%")
print("\nSuggestions:")
for s in body.get("suggestions", []):
    print("  -", s["action"])

## 8. Free cash

`GET /portfolio/cash` — cached snapshot, always instant.  
`POST /portfolio/cash/{slug}/sync` — fetches available cash from `/oms/user/margins` via the cached enctoken (~1 s, no CDP needed).

In [ ]:
# Cached snapshot — instant, no network call
_, snap = get("/portfolio/cash")
entry = next((c for c in snap.get("cash", []) if c["source"] == SLUG), None)
if entry:
    avail = "✓" if entry["cash_available"] else "✗ (not yet synced)"
    print(f"Cached  [{avail}] ₹{entry.get('cash', 0):,.2f}  as_of={entry.get('cash_as_of') or 'never'}")
else:
    print("Source not found in /cash response")

# Live sync via enctoken HTTP (~1 s, no CDP)
print("\nSyncing …")
status, body = post(f"/portfolio/cash/{SLUG}/sync")
print(f"Status: {status}")
if status == 200:
    c = body["cash"]
    print(f"Fresh   [✓] ₹{c.get('cash', 0):>12,.2f}  as_of={c.get('cash_as_of')}")
else:
    pp(body)

## 8. Reset zerodha cache

In [ ]:
from app.modules.brokers import SOURCES

SOURCES[SLUG].reset()
status, body = get(f"/portfolio/sources/{SLUG}")
print(f"{SLUG}: status={body['status']}  holdings={body['holdings_count']}")